### 웹스크래핑 연습문제 2-2

In [ ]:
import os
from urllib.parse import parse_qs, urlparse
from bs4 import BeautifulSoup
import requests


def download_one_episode(title, no, url):
    # 1. 저장할 디렉토리 경로 생성 (img\제목\회차번호)
    dir_path = os.path.join("img", str(title), str(no))
    os.makedirs(dir_path, exist_ok=True)

    # 2. URL에서 titleId 파라미터 추출
    parsed_url = urlparse(url)
    title_id = parse_qs(parsed_url.query).get("titleId", [None])[0]

    # 3. 요청 헤더 설정 (Referer 지정으로 403 차단 방지)
    req_header = {
        "referer": url,
        "user-agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
            " (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        ),
    }

    img_urls = []

    # 4-1. [API 방식] 네이버 웹툰 내부 API로 이미지 URL 수집
    if title_id:
        api_url = f"https://comic.naver.com/api/article/detail?titleId={title_id}&no={no}"
        try:
            api_res = requests.get(api_url, headers=req_header)
            if api_res.ok:
                data = api_res.json()
                image_info_list = (
                    data.get("article", {})
                    .get("contents", {})
                    .get("imageInfo", [])
                )
                for item in image_info_list:
                    img_url = item.get("url")
                    if img_url and img_url not in img_urls:
                        img_urls.append(img_url)
        except Exception:
            pass

    # 4-2. [HTML 파싱 방식] API 실패 시 웹페이지 직접 파싱 (Fallback)
    if not img_urls:
        res = requests.get(url, headers=req_header)
        if res.ok:
            soup = BeautifulSoup(res.content, "html.parser")
            img_tags = soup.select(
                "div.wt_viewer img, img[src*='image-comic.pstatic.net']"
            )
            for img in img_tags:
                src = img.get("src") or img.get("data-src")
                if (
                    src
                    and "image-comic.pstatic.net" in src
                    and src not in img_urls
                ):
                    img_urls.append(src)

    print(
        f"[{title} {no}화] 총 {len(img_urls)}개의 이미지를 다운로드합니다."
    )

    # 5. 이미지 다운로드 및 파일 저장
    for idx, img_url in enumerate(img_urls, start=1):
        img_res = requests.get(img_url, headers=req_header)

        if img_res.ok:
            img_data = img_res.content

            # URL에서 파일명 추출
            file_name = os.path.basename(img_url)

            # img\제목\회차번호\파일명 경로로 지정
            file_path = os.path.join(dir_path, file_name)

            with open(file_path, "wb") as file:
                file.write(img_data)
                print(
                    f"[{idx}/{len(img_urls)}] Writing to {file_path}"
                    f" ({len(img_data):,} bytes)"
                )
        else:
            print(
                f"[{idx}/{len(img_urls)}] 다운로드 실패 (상태 코드:"
                f" {img_res.status_code})"
            )


# 함수 호출 실행
download_one_episode(
    "일렉시드",
    341,
    "https://comic.naver.com/webtoon/detail?titleId=717481&no=341&week=wed",
)